In [1]:
!pip install pandas numpy scikit-learn


In [3]:
# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------

import os
import re
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ------------------------------------------------------------
# 2. SET DATASET PATH
# ------------------------------------------------------------

corpus_path = r"C:\Users\Divya\OneDrive\Desktop\Dokumen\SEM - 5\NLP Skill\Skill Task\Plagiarism_Detector\pan-plagiarism-corpus-2011\intrinsic-detection-corpus\suspicious-document"


# ------------------------------------------------------------
# 3. FIND ALL .TXT DOCUMENTS
# ------------------------------------------------------------

txt_files = []

for root, folders, files in os.walk(corpus_path):

    for file in files:

        # Use only .txt files
        if file.lower().endswith(".txt"):

            file_path = os.path.join(
                root,
                file
            )

            txt_files.append(file_path)


# Sort files
txt_files.sort()


print("============================================")
print("PAN PLAGIARISM CORPUS")
print("============================================")

print(
    "Total TXT documents found:",
    len(txt_files)
)


# ------------------------------------------------------------
# 4. USE A SMALL NUMBER OF DOCUMENTS
# ------------------------------------------------------------
# For the college assignment demonstration,
# we use the first 10 documents.
#
# You can increase this number later.

number_of_documents = 10

txt_files = txt_files[:number_of_documents]


print(
    "Documents selected:",
    len(txt_files)
)


# ------------------------------------------------------------
# 5. LOAD DOCUMENTS
# ------------------------------------------------------------

documents = []

document_names = []


for file_path in txt_files:

    try:

        with open(
            file_path,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as file:

            text = file.read()


        documents.append(text)

        document_names.append(
            os.path.basename(file_path)
        )


    except Exception as e:

        print(
            "Error reading:",
            file_path
        )


print("\nDocuments loaded successfully!")


print("\nDocument names:")

for name in document_names:

    print(name)


# ------------------------------------------------------------
# 6. CLEAN TEXT
# ------------------------------------------------------------

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    # Remove numbers
    text = re.sub(
        r"\d+",
        " ",
        text
    )

    # Remove special characters
    text = re.sub(
        r"[^a-z\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# ------------------------------------------------------------
# 7. CLEAN ALL DOCUMENTS
# ------------------------------------------------------------

cleaned_documents = []

for document in documents:

    cleaned = clean_text(document)

    cleaned_documents.append(cleaned)


print("\nText cleaning completed!")


# ------------------------------------------------------------
# 8. DISPLAY ORIGINAL VS CLEANED TEXT
# ------------------------------------------------------------

print("\n============================================")
print("ORIGINAL VS CLEANED TEXT")
print("============================================")


for i in range(
    min(3, len(documents))
):

    print("\nDocument:",
          document_names[i])

    print("\nOriginal text:")
    print(documents[i][:300])

    print("\nCleaned text:")
    print(cleaned_documents[i][:300])

    print(
        "\n--------------------------------------------"
    )


# ------------------------------------------------------------
# 9. CONVERT TEXT INTO TF-IDF VECTORS
# ------------------------------------------------------------

vectorizer = TfidfVectorizer(
    stop_words="english"
)


tfidf_matrix = vectorizer.fit_transform(
    cleaned_documents
)


print("\n============================================")
print("TF-IDF")
print("============================================")

print(
    "TF-IDF matrix shape:",
    tfidf_matrix.shape
)


# ------------------------------------------------------------
# 10. CALCULATE COSINE SIMILARITY
# ------------------------------------------------------------

similarity_matrix = cosine_similarity(
    tfidf_matrix
)


# ------------------------------------------------------------
# 11. CREATE SIMILARITY MATRIX
# ------------------------------------------------------------

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=document_names,
    columns=document_names
)


print("\n============================================")
print("SIMILARITY MATRIX")
print("============================================")

print(
    similarity_df.round(2)
)


# ------------------------------------------------------------
# 12. SET PLAGIARISM THRESHOLD
# ------------------------------------------------------------

threshold = 0.70


print("\n============================================")
print("PLAGIARISM THRESHOLD")
print("============================================")

print(
    "Threshold:",
    threshold * 100,
    "%"
)


# ------------------------------------------------------------
# 13. COMPARE DOCUMENT PAIRS
# ------------------------------------------------------------

results = []


for i in range(
    len(document_names)
):

    for j in range(
        i + 1,
        len(document_names)
    ):

        score = similarity_matrix[i][j]

        percentage = score * 100


        # Check plagiarism threshold
        if score >= threshold:

            status = "Possible Plagiarism"

        else:

            status = "Low Similarity"


        results.append(
            [
                document_names[i],
                document_names[j],
                round(percentage, 2),
                status
            ]
        )


# ------------------------------------------------------------
# 14. CREATE REPORT DATAFRAME
# ------------------------------------------------------------

report = pd.DataFrame(
    results,
    columns=[
        "Document 1",
        "Document 2",
        "Similarity (%)",
        "Status"
    ]
)


# ------------------------------------------------------------
# 15. RANK BY SIMILARITY
# ------------------------------------------------------------

report = report.sort_values(
    by="Similarity (%)",
    ascending=False
)


# ------------------------------------------------------------
# 16. DISPLAY RANKED REPORT
# ------------------------------------------------------------

print("\n============================================")
print("RANKED SIMILARITY REPORT")
print("============================================")

print(
    report.to_string(index=False)
)


# ------------------------------------------------------------
# 17. DISPLAY SUSPICIOUS PAIRS
# ------------------------------------------------------------

suspicious = report[
    report["Similarity (%)"] >=
    threshold * 100
]


print("\n============================================")
print("SUSPICIOUS DOCUMENT PAIRS")
print("============================================")


if len(suspicious) == 0:

    print(
        "No potentially copied documents found."
    )

else:

    print(
        suspicious.to_string(
            index=False
        )
    )


# ------------------------------------------------------------
# 18. SAVE SIMILARITY MATRIX
# ------------------------------------------------------------

similarity_df.to_csv(
    "similarity_matrix.csv"
)


# ------------------------------------------------------------
# 19. SAVE PLAGIARISM REPORT
# ------------------------------------------------------------

report.to_csv(
    "plagiarism_report.csv",
    index=False
)


# ------------------------------------------------------------
# 20. COMPLETION MESSAGE
# ------------------------------------------------------------

print("\n============================================")
print("PROCESS COMPLETED SUCCESSFULLY!")
print("============================================")

print(
    "\nSimilarity matrix saved as:"
)

print(
    "similarity_matrix.csv"
)

print(
    "\nPlagiarism report saved as:"
)

print(
    "plagiarism_report.csv"
)

PAN PLAGIARISM CORPUS
Total TXT documents found: 4753
Documents selected: 10

Documents loaded successfully!

Document names:
suspicious-document04501.txt
suspicious-document04502.txt
suspicious-document04503.txt
suspicious-document04504.txt
suspicious-document04505.txt
suspicious-document04506.txt
suspicious-document04507.txt
suspicious-document04508.txt
suspicious-document04509.txt
suspicious-document04510.txt

Text cleaning completed!

ORIGINAL VS CLEANED TEXT

Document: suspicious-document04501.txt

Original text:
﻿Yes, but open your eyes again and you will see it in the same place, in the same form, doing
the same work. A most persistent nothing, a most powerful nothing! Not the shadow cast by the
good, but the cloud that hides the sun and casts the shadow. Not the "silence implying sound,"
but the discord b

Cleaned text:
yes but open your eyes again and you will see it in the same place in the same form doing the same work a most persistent nothing a most powerful nothing not th